In [ ]:
# =============================================
# IMPORT
# =============================================
from utility.logger         import aiLogger
from keras.callbacks        import EarlyStopping, ReduceLROnPlateau
from model.componentModel   import mainModel
from preProcessingDataTEP   import PreProcessData
from prettytable            import PrettyTable

from keras.optimizers       import Adam
from utility.matpilotDraw   import MatPlot
from sklearn.preprocessing  import MinMaxScaler

import tensorflow           as tf
import pandas               as pd
import matplotlib.pyplot    as plt
import numpy                as np
import os, warnings
import random
import json


In [2]:
# ---------------------------
# Loading config file
# ---------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    configData = json.load(configFile)
config = configData["tep"]  

# ---------------------------
# Set debug level and 
# ---------------------------
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

outputDir           = os.path.expanduser(config["dataDir"])

#### Preprocessing data

In [ ]:
# ==============================
# Config and data preprocessing
# ==============================
preData             = PreProcessData(config=config)
rawData             = preData.loadDataFromH5Format(dir = outputDir, name= "data/teps.h5", case="all")



withRULData         = preData.addRULAndEnvironment(rawData)
withoutInitRawData  = preData.removeInitState(inData= withRULData)



# ---------------------------
# For data
cleanData           = preData.cleanData(rawSignal           =   withoutInitRawData,
                                        columnName          =   preData.allWithoutLabel,
                                        getMovingAverage    =   False,
                                        flipToDown          =   False)
# ---------------------------
# For label, flip only
cleanData           = preData.cleanData(rawSignal           =   cleanData,
                                        columnName          =   preData.degraLiq,
                                        getMovingAverage    =   False,
                                        flipToDown          =   False)
norData             = preData.normalizeDataEachModes(rawData    = cleanData,
                                                     headline   =   preData.headline["allWithoutLabelNoBias"] + ["rul"])                  # Normalize the data

# test = withoutInitRawData[1]
# print(test.head)
# print(test[test["Id"] == 1].shape)
# ---------------------------
# Extract the data in each Case
# data                = norData[config["case"]]
# data                = norData[2]
# trajectID           = data["Id"]
# reactor             = data[preData.headline["reactor"]]
# separator           = data[preData.headline["separator"]]
# stripper            = data[preData.headline["stripper"]]
# allSensor           = data[preData.headline["allWithoutLabelNoBias"]]
# rul                 = data["rul"]
# env                 = data["environmentID"]
# timeTraj            = data["Time"]


# allSeqRul = []
# for idx in range(100):
#     getAtIdx = idx + 1
#     print(f"Getting the HI at the idx {getAtIdx}")
#     seqRul, seqXAllSensorTrain     = preData.getTensorData( dataList    = [rul,allSensor],
#                                                             trajectId    = trajectID,
#                                                             getAtIdx    = getAtIdx,   
#                                                                     )
#     allSeqRul.append(seqRul)


#### Declare the model

In [ ]:
config["component"] = "Reac"
vaeReac = mainModel(config = config)
vaeReac.build(input_shape=[(None, config["seqLength"], config["inReac"]), (None, 1, config["inReac"]), (None, 1)])
vaeReac.summary()
vaeReac.load_weights(outputDir + "checkpoint/bestModel/")

In [5]:
# config["component"] = "Sep"
# vaeSep = mainModel(config = config)
# vaeSep.build(input_shape=[(None, config["seqLength"], config["inSep"]), (None, 1, config["inSep"]), (None, 1)])
# vaeSep.summary()
# vaeSep.load_weights(outputDir + "checkpoint/bestModel22Sep/")

In [6]:
# config["component"] = "Strip"
# vaeStrip = mainModel(config = config)
# vaeStrip.build(input_shape=[(None, config["seqLength"], config["inStrip"]), (None, 1, config["inStrip"]), (None, 1)])
# vaeStrip.summary()
# vaeStrip.load_weights(outputDir + "checkpoint/bestModel23Strip/")

### Environment 1

In [ ]:
environmentIdx      = 2
data                = norData[environmentIdx]
trajectID           = data["Id"]
allSensor           = data[preData.headline["allWithoutLabelNoBias"]]
allSensor           = data[preData.headline["reactor"]]
rul                 = data["rul"]
env                 = data["environmentID"]
timeTraj            = data["Time"]

hiEnv1              = {}
rulEnv1             = {}
hiEnv1["Reac"]      = []
rulEnv1["Reac"]     = []
hiEnv1["Sep"]       = []
rulEnv1["Sep"]      = []
hiEnv1["Strip"]     = []
rulEnv1["Strip"]    = []

timeEachCase        = withoutInitRawData[environmentIdx].groupby('Id',sort=False)['Time']
trajecTime          = [eachTrajectTime[1] for eachTrajectTime in timeEachCase]                  #Extract time from the tuple and groupby
# for eachTrajectTime in timeEachCase:
#     print(len(eachTrajectTime)) 

for idx in range(2):
    getAtIdx = idx + 1
    print(f"Getting the HI at the idx {getAtIdx}")
    seqXAllSensorTrain,     \
    seqXRulTrain,           \
    seqXTrainEnv,seqTime                = preData.getTensorData(dataList    = [allSensor,rul,env,timeTraj],
                                                                trajectId   = trajectID,  
                                                                shuffle     = True,      
                                                                getAtIdx    = idx + 1,
                                                                seqLength   = preData.config["seqLength"],
                                                                overlap     = preData.config["overLap"]         
                                                                )
    seqTimeTrain        = seqTime[:,-1,:]

    xPredict            = [seqXAllSensorTrain, seqTime[:,-1,:], seqXTrainEnv[:,-1,:]]  
    # print(seqXAllSensorTrain.shape)  
    hiReac, binaryMask,probs, rulReac               = vaeReac.getHIandRUL(input     = xPredict)
    # hiSep, binaryMask,probs, rulSep                 = vaeSep.getHIandRUL(input      = xPredict)
    # hiStrip, binaryMask,probs, rulStrip             = vaeStrip.getHIandRUL(input    = xPredict)
    # squeezedHiSystem                    = tf.reduce_mean(hiSystem,axis=-1)
    # print(squeezedHiSystem.shape)
    squeezedRulReac                                 = tf.squeeze(rulReac, axis=-1)
    # squeezedRulSep                                  = tf.squeeze(rulSep, axis=-1)
    # squeezedRulStrip                                = tf.squeeze(rulStrip, axis=-1)

    
    hiEnv1["Reac"].append(hiReac)
    # hiEnv1["Sep"].append(hiSep)
    # hiEnv1["Strip"].append(hiStrip)

    rulEnv1["Reac"].append(squeezedRulReac)
    # rulEnv1["Sep"].append(squeezedRulSep)
    # rulEnv1["Strip"].append(squeezedRulStrip)




In [8]:
# print(binaryMask[100])
# ssTable                 = PrettyTable()
# ssTable.field_names     = ["No","sensorName","probability", "binary"]
# for idxEnv in range(3):
#     print(f"The number of selected sensor {sum(binaryMask[idxEnv])}")
#     for idx,idxSS in enumerate(preData.headline["allWithoutLabelNoBias"]):
#         ssTable.add_row([idx,idxSS,binaryMask[idx].numpy(),binaryMask[idx].numpy()])

# print(ssTable)

In [ ]:
plot1 = MatPlot() 
plot1.draw1D(hiEnv1["Reac"][0])

In [ ]:
print(hiEnv1["Reac"][0].shape)

In [ ]:
plot1 = MatPlot() 
plot1.subPlotMultiImg(inY = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"]], color=['g','b', 'y'], numRow=1, numCol=2)

In [ ]:
plot1 = MatPlot() 
plot1.subPlotMultiImg(inY = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"]], color=['g','b', 'y'], numRow=30, numCol=5)

### Environment 2

In [ ]:
environmentIdx      = 2
data                = norData[environmentIdx]
trajectID           = data["Id"]
allSensor           = data[preData.headline["allWithoutLabelNoBias"]]
rul                 = data["rul"]
env                 = data["environmentID"]
timeTraj            = data["Time"]

hiEnv1              = {}
rulEnv1             = {}
hiEnv1["Reac"]      = []
rulEnv1["Reac"]     = []
hiEnv1["Sep"]       = []
rulEnv1["Sep"]      = []
hiEnv1["Strip"]     = []
rulEnv1["Strip"]    = []

timeEachCase        = withoutInitRawData[environmentIdx].groupby('Id',sort=False)['Time']
trajecTime          = [eachTrajectTime[1] for eachTrajectTime in timeEachCase]                  #Extract time from the tuple and groupby
# for eachTrajectTime in timeEachCase:
#     print(len(eachTrajectTime)) 

for idx in range(150):
    getAtIdx = idx + 1
    print(f"Getting the HI at the idx {getAtIdx}")
    seqXAllSensorTrain,     \
    seqXRulTrain,           \
    seqXTrainEnv,seqTime                = preData.getTensorData(dataList    = [allSensor,rul,env,timeTraj],
                                                                trajectId   = trajectID,  
                                                                shuffle     = True,      
                                                                seqLength   = preData.config["seqLength"],
                                                                overlap     = preData.config["overLap"]         
                                                                )
    seqTimeTrain        = seqTime[:,-1,:]

    xPredict            = [seqXAllSensorTrain, seqTime[:,-1,:], seqXTrainEnv[:,-1,:]]    
    hiReac, binaryMask,probs, rulReac               = vaeReac.getHIandRUL(input     = xPredict, comp = "Reac")
    hiSep, binaryMask,probs, rulSep                 = vaeSep.getHIandRUL(input      = xPredict, comp = "Sep")
    hiStrip, binaryMask,probs, rulStrip             = vaeStrip.getHIandRUL(input    = xPredict, comp = "Strip")
    # squeezedHiSystem                    = tf.reduce_mean(hiSystem,axis=-1)
    # print(squeezedHiSystem.shape)
    squeezedRulReac                                 = tf.squeeze(rulReac, axis=-1)
    squeezedRulSep                                  = tf.squeeze(rulSep, axis=-1)
    squeezedRulStrip                                = tf.squeeze(rulStrip, axis=-1)

    
    hiEnv1["Reac"].append(hiReac)
    hiEnv1["Sep"].append(hiSep)
    hiEnv1["Strip"].append(hiStrip)

    rulEnv1["Reac"].append(squeezedRulReac)
    rulEnv1["Sep"].append(squeezedRulSep)
    rulEnv1["Strip"].append(squeezedRulStrip)




In [ ]:
plot1 = MatPlot() 
plot1.subPlotMultiImg(inY = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"]], color=['g','b', 'y'], numRow=30, numCol=5)

In [ ]:
environmentIdx      = 3
data                = norData[environmentIdx]
trajectID           = data["Id"]
allSensor           = data[preData.headline["allWithoutLabelNoBias"]]
rul                 = data["rul"]
env                 = data["environmentID"]
timeTraj            = data["Time"]

hiEnv1              = {}
rulEnv1             = {}
hiEnv1["Reac"]      = []
rulEnv1["Reac"]     = []
hiEnv1["Sep"]       = []
rulEnv1["Sep"]      = []
hiEnv1["Strip"]     = []
rulEnv1["Strip"]    = []

timeEachCase        = withoutInitRawData[environmentIdx].groupby('Id',sort=False)['Time']
trajecTime          = [eachTrajectTime[1] for eachTrajectTime in timeEachCase]                  #Extract time from the tuple and groupby
# for eachTrajectTime in timeEachCase:
#     print(len(eachTrajectTime)) 

for idx in range(150):
    getAtIdx = idx + 1
    print(f"Getting the HI at the idx {getAtIdx}")
    seqXAllSensorTrain,     \
    seqXRulTrain,           \
    seqXTrainEnv,seqTime                = preData.getTensorData(dataList    = [allSensor,rul,env,timeTraj],
                                                                trajectId   = trajectID,  
                                                                shuffle     = True,      
                                                                seqLength   = preData.config["seqLength"],
                                                                overlap     = preData.config["overLap"]         
                                                                )
    seqTimeTrain        = seqTime[:,-1,:]

    xPredict            = [seqXAllSensorTrain, seqTime[:,-1,:], seqXTrainEnv[:,-1,:]]    
    hiReac, binaryMask,probs, rulReac               = vaeReac.getHIandRUL(input     = xPredict, comp = "Reac")
    hiSep, binaryMask,probs, rulSep                 = vaeSep.getHIandRUL(input      = xPredict, comp = "Sep")
    hiStrip, binaryMask,probs, rulStrip             = vaeStrip.getHIandRUL(input    = xPredict, comp = "Strip")
    # squeezedHiSystem                    = tf.reduce_mean(hiSystem,axis=-1)
    # print(squeezedHiSystem.shape)
    squeezedRulReac                                 = tf.squeeze(rulReac, axis=-1)
    squeezedRulSep                                  = tf.squeeze(rulSep, axis=-1)
    squeezedRulStrip                                = tf.squeeze(rulStrip, axis=-1)

    
    hiEnv1["Reac"].append(hiReac)
    hiEnv1["Sep"].append(hiSep)
    hiEnv1["Strip"].append(hiStrip)

    rulEnv1["Reac"].append(squeezedRulReac)
    rulEnv1["Sep"].append(squeezedRulSep)
    rulEnv1["Strip"].append(squeezedRulStrip)




In [ ]:
plot1 = MatPlot() 
plot1.subPlotMultiImg(inY = [hiEnv1["Reac"], hiEnv1["Sep"], hiEnv1["Strip"]], color=['g','b', 'y'], numRow=30, numCol=5)

In [ ]:
# plot1 = MatPlot()
# plot1.subPlotMultiImg(inY = [result["reactor"],result["separator"],result["stripper"]], color=['g','y','b'], numRow=20, numCol=5)

# plot3 = MatPlot()
# plot3.subPlotMultiImg(inY = [result["system"]], color=['r'], numRow=20, numCol=5)

# # plot1.subPlotMultiImg(inY = [scaledHiReac,scaledHiSep,scaledHiStrip, scaledHiSystem], color=['g','y','b','r'], numRow=20, numCol=5)

# plot2 = MatPlot()
# plot2.subPlotMultiImg(inY = [result["rul"]],  numRow=20, numCol=5)

